In [1]:
#from __future__ import annotations
from datetime import date
from typing import Optional, List
from sqlmodel import SQLModel, Field, Relationship, Session, create_engine, select
from sqlalchemy import UniqueConstraint, func

In [2]:
# --- existing ---
class Person(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    name: str = Field(index=True)
    surname: str = Field(index=True)
    age: int
    __table_args__ = (UniqueConstraint("name", "surname", "age", name="uq_person_identity"),)

    # relationship to the link table
    tool_links: list['PersonToolLink'] = Relationship(
        back_populates="person",
        sa_relationship_kwargs={"cascade": "all, delete-orphan"},
    )


In [3]:
class Tool(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    name: str = Field(index=True)              # add unique=True if you want one row per tool name
    __table_args__ = (UniqueConstraint("name", name="uq_tool_name"),)

    # relationship to the link table
    person_links: list['PersonToolLink'] = Relationship(back_populates="tool")

In [4]:
class PersonToolLink(SQLModel, table=True):
    # composite PK prevents duplicate pairs (no second link for same person+tool)
    person_id: int = Field(foreign_key="person.id", primary_key=True)
    tool_id: int   = Field(foreign_key="tool.id", primary_key=True)

    # optional metadata about the skill
    level: int = Field(default=1)              # e.g., 1–5
    certified: bool = False
    since: Optional[date] = None

    person: Person = Relationship(back_populates="tool_links")
    tool: Tool     = Relationship(back_populates="person_links")


In [5]:
engine = create_engine("sqlite:///people.db", echo=False)
SQLModel.metadata.create_all(engine)

In [6]:
def add_person(session: Session, name: str, surname: str, age: int) -> None:
    session.add(Person(name=name, surname=surname, age=age))
    session.commit()

In [7]:
def list_people(session: Session):
    rows = session.exec(select(Person).order_by(Person.id)).all()
    if not rows:
        print("(empty)")
        return
    for p in rows:
        # As dict (Pydantic): p.model_dump()
        print(f"{p.id:3}  {p.name} {p.surname}  age={p.age}")

In [8]:
def count_people(session: Session) -> int:
    n = session.exec(select(func.count(Person.id))).first()
    return n or 0


In [9]:
with Session(engine) as s:
    # add a few (duplicates will fail thanks to the unique constraint)
    try:
        add_person(s, "Makar", "Saviak", 22)
        add_person(s, "Ada", "Lovelace", 36)
    except Exception as e:
        s.rollback()
        print("Insert failed:", e)

    print("All people:")
    list_people(s)

    print("Total:", count_people(s))

    # Filter examples
    print("\nOnly Makar:")
    makar = s.exec(
        select(Person).where(Person.name == "Makar", Person.surname == "Saviak")
    ).first()
    print(makar.model_dump() if makar else "(not found)")

Insert failed: (sqlite3.IntegrityError) UNIQUE constraint failed: person.name, person.surname, person.age
[SQL: INSERT INTO person (name, surname, age) VALUES (?, ?, ?)]
[parameters: ('Makar', 'Saviak', 22)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
All people:
  1  Makar Saviak  age=22
  2  Ada Lovelace  age=36
Total: 2

Only Makar:
{'name': 'Makar', 'surname': 'Saviak', 'age': 22, 'id': 1}


In [10]:
with Session(engine) as s:
    obj1 = Person(name="Vali", surname="Thoss", age=27)
    obj2 = Tool(name="korb")

    for obj in [obj1, obj2]:
        try:
            s.add(obj); s.commit(); s.refresh(obj)
        except Exception as e:
            s.rollback()
            print("Insert failed:", e)


    # make the link row (with metadata)
    link = PersonToolLink(person_id=obj1.id, tool_id=obj2.id,
                          level=4, certified=True)

    s.add(link)
    s.commit()


From now on the real stuff

In [1]:
from datetime import date
from typing import Optional
from sqlmodel import SQLModel, Field, Relationship, Session, create_engine, select
from sqlalchemy import UniqueConstraint, func, event
from pydantic import PositiveFloat, computed_field
import numpy as np

In [2]:
class Syringe(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    nominal_volume_ul: PositiveFloat
    name: str = Field(index=True)
    inner_diameter_mm: PositiveFloat
    __table_args__ = (UniqueConstraint("nominal_volume_ul",
                                       "name",
                                       "inner_diameter_mm",
                                       name="uq_syringe_identity"),)

    # relationship to the link table
    solvent_links: list['SyringeSolventLink'] = Relationship(
        back_populates="syringe",
        sa_relationship_kwargs={"cascade": "all, delete-orphan"},
    )

    @computed_field(return_type=float)
    @property
    def theoretical_correlation_factor(self) -> float:
        """Calculate correlation factor on demand [mm/µL]."""
        r = float(self.inner_diameter_mm) / 2.0
        return 1.0 / (np.pi * r ** 2)

In [3]:
# from sqlalchemy import text
# from sqlmodel import create_engine
#
# engine = create_engine("sqlite:///liquid_handling.db")
# # better do it manually with dbeaver
# with engine.begin() as conn:
#     conn.execute(text('ALTER TABLE syringe RENAME COLUMN nominal_volume TO nominal_volume_ul'))
#     conn.execute(text('ALTER TABLE syringe RENAME COLUMN inner_diameter TO inner_diameter_mm'))
#     conn.execute(text('ALTER TABLE solvent RENAME COLUMN density TO density_g_per_ml'))
#

In [3]:
class Solvent(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    name: str = Field(index=True)
    density_g_per_ml: Optional[PositiveFloat] = Field(default=None)
    notes: Optional[str] = Field(default=None)
    __table_args__ = (UniqueConstraint("name",
                                       "density_g_per_ml",
                                       "notes",
                                       name="uq_solvent_identity"),)

    # relationship to the link table
    syringe_links: list['SyringeSolventLink'] = Relationship(
        back_populates="solvent"
    )

In [4]:
class SyringeSolventLink(SQLModel, table=True):
    # composite PK prevents duplicate pairs (no second link for same person+tool)
    syringe_id: int = Field(foreign_key="syringe.id", primary_key=True)
    solvent_id: int = Field(foreign_key="solvent.id", primary_key=True)

    # optional metadata about the liquid handling
    calibrated: bool = False
    backlash_correction: PositiveFloat = Field(default=0.0)
    real_correlation_factor: Optional[float] = Field(default=None)
    since: Optional[date] = None

    syringe: Syringe = Relationship(back_populates="solvent_links")
    solvent: Solvent = Relationship(back_populates="syringe_links")

    @classmethod
    def __declare_last__(cls):
        @event.listens_for(cls, "before_insert")
        def _fill_factor_on_insert(_mapper, _connection, target: "SyringeSolventLink"):
            if target.real_correlation_factor is None and getattr(target, "syringe", None) is not None:
                target.real_correlation_factor = target.syringe.theoretical_correlation_factor

In [5]:
engine = create_engine("sqlite:///liquid_handling.db", echo=False)
SQLModel.metadata.create_all(engine)

In [6]:
with Session(engine) as s:
    obj1 = Syringe(
        nominal_volume_ul=1000,
        name="Hamilton1001",
        inner_diameter_mm=4.61
        )
    print(f"{obj1.theoretical_correlation_factor:.6f}")
    obj2 = Solvent(
        name="Acetonitrile",
        density_g_per_ml="0.786",
        notes="HPLC grade"
        )

    for obj in [obj1, obj2]:
        try:
            s.add(obj); s.commit(); s.refresh(obj)
        except Exception as e:
            s.rollback()
            print("Insert failed:", e)

    print(obj1)
    # make the link row (with metadata)
    link = SyringeSolventLink(syringe=obj1, solvent=obj2,
                          calibrated=False)
    s.add(link)
    s.commit()

0.059911
name='Hamilton1001' id=1 inner_diameter_mm=4.61 nominal_volume_ul=1000.0 theoretical_correlation_factor=0.059911234406725106


In [8]:
engine.dispose()

In [25]:
with Session(engine) as s:
    syringe = s.get(Syringe, 1)  # fetch by primary key
    if syringe:
        print(syringe)
        print(type(syringe))
    else:
        print("No syringe with this ID.")


id=1 name='Hamilton1001' nominal_volume_ul=1000.0 inner_diameter_mm=4.61 theoretical_correlation_factor=0.059911234406725106
<class '__main__.Syringe'>


In [7]:
with Session(engine) as s:
    syringe = s.get(Syringe, 1)
    ssl = s.get(SyringeSolventLink, (1,1))  # fetch by primary key
    if ssl:
        print(ssl)
        print(type(ssl))
    else:
        print("No ssl with this ID.")

syringe_id=1 backlash_correction=0.0 since=None calibrated=False solvent_id=1 real_correlation_factor=None
<class '__main__.SyringeSolventLink'>
